# Music Annotation Project V8.7
## Notebook 51R2: Frozen M3 vs M5-REMOVE vs diagnostic M4 Reconstruction Crossover

**Development-only. No production or promotion authority.**

This replacement corrects the previous Notebook 51 lineage/model-loading errors.

### Fixed experiment
- Mask arms: **M3**, **M5**, **M4_DIAGNOSTIC**.
- `M5 = M3 minus Notebook 50 SAFE_REMOVE plus Notebook 50 SAFE_PROTECT.`
- Restorers: **R1_biharmonic**, **R2_telea**, **R3_learned_unet**.
- R3 uses Notebook 46R6's exact `TinyUNet` architecture and the checkpoint for the page's held-out `source_group`.
- M3 and M4 whole-page candidates are reused from the completed Notebook 46R6 run where available. This prevents accidental restorer drift.
- M5-REMOVE is newly reconstructed under the same frozen restorer implementations.
- M4 remains diagnostic only.
- Clean truth is unavailable until the complete candidate inventory is written and hashed.
- Outside the authoritative mask is byte-for-byte immutable.
- Paired page and source-group analyses are mandatory.
- A secondary endpoint tests whether the between-restorer gap contracts from M3 to M5-REMOVE.

**Important:** Notebook 46R6 does not create `experiment_identity.json`. This notebook instead verifies its actual durable identity set: `source_record.json`, `restorer_pre_registration.json`, `v8_6_verdict.json`, `audit/output_completion.json`, and `audit/learned_model_inventory.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image matplotlib torch torchvision


## 1. Configuration, output tree and pre-registration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, zipfile, gc, time, os
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
N46_ROOT=ROOT/'observed_print_classifier_v8_6_notebook46'
N50_ROOT=ROOT/'observable_residual_mask_m5_v8_7'
OUTROOT=ROOT/'observed_print_classifier_v8_7_notebook51'

N45_RUN_OVERRIDE=None
N46_RUN_OVERRIDE=None
N50_RUN_OVERRIDE=None
RESUME_RUN_OVERRIDE=None
AUTO_RESUME_LATEST_INCOMPLETE=True
ALLOW_NEW_RUN_IF_NONE_FOUND=True

SEED=51002
BOOT=10000
PATCH=128
CROP_MARGIN=48
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 51 R3 learned restorer requires a GPU runtime.')

ARMS=['M3','M5','M4_DIAGNOSTIC']
RESTORERS=['R1_biharmonic','R2_telea','R3_learned_unet']
PRIMARY_RESTORER='R3_learned_unet'


def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def atomic_csv(frame,path):
    path=Path(path); tmp=Path(str(path)+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(path)

def require(condition,message):
    if not condition:
        if 'AUDIT' in globals(): atomic_json(AUDIT/'fatal_failure.json',{'utc':datetime.now(timezone.utc).isoformat(),'message':message})
        raise RuntimeError(message)

def newest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

# Resume only Notebook51R2 directories carrying this pre-registration.
def resumable_runs(root):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and (p/'01_pre_registration.json').exists():
                try:
                    pre=json.loads((p/'01_pre_registration.json').read_text())
                    if pre.get('version')=='8.7-notebook51R2': hits.append(p)
                except Exception: pass
    return sorted(hits,key=lambda p:p.stat().st_mtime,reverse=True)

if RESUME_RUN_OVERRIDE:
    OUT=Path(RESUME_RUN_OVERRIDE); RESUME=True
elif AUTO_RESUME_LATEST_INCOMPLETE:
    rr=resumable_runs(OUTROOT); OUT=rr[0] if rr else None; RESUME=OUT is not None
else:
    OUT=None; RESUME=False
if OUT is None:
    require(ALLOW_NEW_RUN_IF_NONE_FOUND,'No resumable Notebook51R2 run found.')
    OUT=OUTROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_7_notebook51R2'); RESUME=False

INPUT=OUT/'01_source_lock'; CHECK=OUT/'02_page_checkpoints'; CAND=OUT/'03_candidates'; LOCK=OUT/'04_candidate_lock'; TABLES=OUT/'05_tables'; VIS=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUT,CHECK,CAND,LOCK,TABLES,VIS,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=True)

PREREG={'version':'8.7-notebook51R2','development_only':True,'mask_arms':ARMS,'restorers':RESTORERS,'primary_restorer':PRIMARY_RESTORER,'m5_definition':'M3 minus Notebook50 SAFE_REMOVE only','safe_protect_intervention':False,'m4_role':'diagnostic only','truth_firewall':'clean truth loaded only after candidate lock','bootstrap_resamples':BOOT,'promotion_authority':False}
pre_path=OUT/'01_pre_registration.json'
if pre_path.exists(): require(json.loads(pre_path.read_text())==PREREG,'Resume blocked: Notebook51R2 pre-registration changed.')
else: atomic_json(pre_path,PREREG)
print('Mode:', 'RESUME' if RESUME else 'NEW RUN'); print('Output:',OUT); print('GPU:',torch.cuda.get_device_name(0))


## 2. Resolve exact completed Notebook 45, 46R6 and 50 runs


In [ ]:
REQ45=['mask_lock.json','source_record.json','tables/region_manifest.parquet']
REQ46=['source_record.json','restorer_pre_registration.json','v8_6_verdict.json','audit/output_completion.json','audit/learned_model_inventory.csv','tables/whole_page_candidate_inventory.csv']
REQ50=['notebook50_verdict.json','tables/m5_map_inventory.csv']

N45=Path(N45_RUN_OVERRIDE) if N45_RUN_OVERRIDE else newest_complete(N45_ROOT,REQ45)
N46=Path(N46_RUN_OVERRIDE) if N46_RUN_OVERRIDE else newest_complete(N46_ROOT,REQ46)
N50=Path(N50_RUN_OVERRIDE) if N50_RUN_OVERRIDE else newest_complete(N50_ROOT,REQ50)
require(N45 is not None,'No complete Notebook45 run found. Set N45_RUN_OVERRIDE.')
require(N46 is not None,'No complete Notebook46R6 run found. Set N46_RUN_OVERRIDE.')
require(N50 is not None,'No complete Notebook50 run found. Set N50_RUN_OVERRIDE.')

lock45=json.loads((N45/'mask_lock.json').read_text())
source45=json.loads((N45/'source_record.json').read_text())
source46=json.loads((N46/'source_record.json').read_text())
pre46=json.loads((N46/'restorer_pre_registration.json').read_text())
verdict46=json.loads((N46/'v8_6_verdict.json').read_text())
complete46=json.loads((N46/'audit/output_completion.json').read_text())
verdict50=json.loads((N50/'notebook50_verdict.json').read_text())

require(verdict46.get('valid_run') is True,'Notebook46R6 verdict is not valid_run=True.')
require(complete46.get('complete') is True,'Notebook46R6 output_completion is not complete.')
require('R3_learned_unet' in pre46.get('restorers',[]),'Notebook46R6 pre-registration lacks R3_learned_unet.')
require(pre46.get('mask_lock_id')==source46.get('mask_lock_id')==lock45.get('lock_id'),'Notebook45/46 mask-lock lineage mismatch.')
require(verdict50.get('verdict')=='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER','Notebook50 verdict does not authorise REMOVE-only crossover.')

regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
need={'page','source_group','region_id','y0','y1','x0','x1'}
require(need.issubset(regions.columns),f'Notebook45 region manifest missing {sorted(need-set(regions.columns))}')
regions['page']=regions.page.astype(str); regions['source_group']=regions.source_group.astype(str)
require(not regions[['page','region_id']].duplicated().any(),'Duplicate Notebook45 region identity.')

source_record={'notebook45':str(N45),'notebook46':str(N46),'notebook50':str(N50),'mask_lock_id':lock45.get('lock_id'),'notebook46_verdict':verdict46.get('decision'),'notebook50_verdict':verdict50.get('verdict')}
old_path=OUT/'source_record.json'
if old_path.exists(): require(json.loads(old_path.read_text())==source_record,'Resume blocked: source lineage changed.')
else: atomic_json(old_path,source_record)

for p in [N45/'mask_lock.json',N45/'source_record.json',N46/'source_record.json',N46/'restorer_pre_registration.json',N46/'v8_6_verdict.json',N46/'audit/output_completion.json',N46/'audit/learned_model_inventory.csv',N50/'notebook50_verdict.json',N50/'tables/m5_map_inventory.csv']:
    dst=INPUT/(p.parent.name+'__'+p.name); shutil.copy2(p,dst)
print(json.dumps(source_record,indent=2)); print('Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique(),'Regions:',len(regions))


## 3. Exact Notebook 46R6 TinyUNet and held-out checkpoint resolver


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=nn.Sequential(nn.Conv2d(2,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,32,3,padding=1),nn.ReLU())
        self.p=nn.MaxPool2d(2)
        self.e2=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.Conv2d(64,64,3,padding=1),nn.ReLU())
        self.u=nn.ConvTranspose2d(64,32,2,2)
        self.d=nn.Sequential(nn.Conv2d(64,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,1,1),nn.Sigmoid())
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.p(a)); return self.d(torch.cat([self.u(b),a],1))

def load_n46_model(path):
    model=TinyUNet().to(DEVICE); state=torch.load(path,map_location=DEVICE); model.load_state_dict(state); model.eval()
    with torch.no_grad(): y=model(torch.zeros((1,2,PATCH,PATCH),dtype=torch.float32,device=DEVICE))
    require(tuple(y.shape)==(1,1,PATCH,PATCH) and torch.isfinite(y).all(),f'Checkpoint smoke test failed: {path}')
    return model

mi=pd.read_csv(N46/'audit/learned_model_inventory.csv')
required={'test_source_group','checkpoint','sha256'}; require(required.issubset(mi.columns),f'Notebook46 learned model inventory missing {sorted(required-set(mi.columns))}')
checkpoint_by_group={}
for _,r in mi.iterrows():
    g=str(r.test_source_group); p=Path(str(r.checkpoint))
    if not p.exists():
        # Safe recovery if the Notebook46 run was moved but models stayed in its models folder.
        alt=N46/'models'/p.name
        require(alt.exists(),f'Missing Notebook46 checkpoint for {g}: {p}'); p=alt
    require(sha(p)==str(r.sha256),f'Notebook46 checkpoint hash mismatch for {g}: {p}')
    checkpoint_by_group[g]=p
missing_groups=sorted(set(regions.source_group)-set(checkpoint_by_group)); require(not missing_groups,f'No held-out Notebook46 model for source groups: {missing_groups}')
print('Verified held-out TinyUNet checkpoints:',len(checkpoint_by_group))


## 4. Page inputs, frozen M3/M4 masks and Notebook 46 whole-page bases


In [ ]:
source_v82=Path(source45['source_v82'])
clean_dir=ROOT/'source'/'clean'
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
clean_map={p.stem:p for p in clean_dir.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
pages=sorted(regions.page.unique())
page_obs={}; page_base={}; page_auth={}; page_protected={}; page_masks={}

def union_arm(page,arm):
    arr=None
    for _,r in regions[regions.page==page].iterrows():
        p=N45/'masks'/page/str(r.region_id).replace(':','_')/f'{arm}.png'; m=cv2.imread(str(p),0)
        require(m is not None,f'Missing/unreadable Notebook45 mask {p}'); b=m>0; arr=b.copy() if arr is None else (arr|b)
    return arr

for page in pages:
    print(page)
    obs=cv2.imread(str(source_v82/'whole_pages'/page/'observed.png'),0)
    base=cv2.imread(str(N46/'whole_page_candidates'/page/'v7_8.png'),0)
    protected=cv2.imread(str(N46/'whole_page_candidates'/page/'protected_print.png'),0)
    if base is None: base=cv2.imread(str(source_v82/'whole_pages'/page/'v7_8.png'),0)
    require(obs is not None and base is not None and protected is not None,f'{page}: missing observed/base/protected source page.')
    v75=Path(source45['v75']); status=pd.read_csv(v75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
    require(page in cache,f'{page}: missing V7.5 cache mapping.')
    with np.load(cache[page]) as z: auth=z['mask']>0
    require(obs.shape==base.shape==protected.shape==auth.shape,f'{page}: page source shape mismatch.')
    require(np.array_equal(base[~auth],obs[~auth]),f'{page}: V7.8 outside-authority drift.')
    page_obs[page]=obs; page_base[page]=base; page_auth[page]=auth; page_protected[page]=protected>0
    page_masks[page]={'M3':union_arm(page,'M3'),'M4_DIAGNOSTIC':union_arm(page,'M4')}
print('Loaded page bases and frozen M3/M4 unions:',len(pages))


## 5. Resolve Notebook 50 SAFE_REMOVE maps and construct M5-REMOVE


In [ ]:
inv50 = pd.read_csv(N50/'tables/m5_map_inventory.csv')
require('page' in inv50.columns, 'Notebook50 m5_map_inventory.csv lacks page.')
require('file' in inv50.columns, 'Notebook50 m5_map_inventory.csv lacks file.')
inv50['page'] = inv50.page.astype(str)

def resolve_m5_decisions(page):
    q = inv50[inv50.page == page]
    require(len(q) == 1, f'{page}: expected exactly one Notebook50 M5 inventory row, found {len(q)}.')

    p = Path(q.iloc[0]['file'])
    require(p.exists(), f'{page}: Notebook50 M5 decision file missing: {p}')
    require(p.suffix.lower() == '.npz', f'{page}: expected Notebook50 M5 decision NPZ, got: {p}')

    with np.load(p) as z:
        require(
            'm5_decision_map' in z.files,
            f'{page}: {p.name} lacks m5_decision_map.'
        )
        decision_small = z['m5_decision_map']

    m3 = page_masks[page]['M3']

    require(
        decision_small.ndim == 2,
        f'{page}: M5 decision map must be 2-D, got {decision_small.shape}.'
    )

    # Notebook50's saved decision canvas may terminate at the
    # maximum M3 coordinate rather than at the physical page edge.
    # It may therefore be smaller than the authoritative page/M3 canvas,
    # but it must never exceed it.
    require(
        decision_small.shape[0] <= m3.shape[0]
        and decision_small.shape[1] <= m3.shape[1],
        f'{page}: Notebook50 decision map {decision_small.shape} exceeds '
        f'authoritative M3/page shape {m3.shape}.'
    )

    # Restore Notebook50 decisions onto the authoritative page canvas.
    # Zero means no assertion / FALLBACK_M3.
    decision = np.zeros(m3.shape, dtype=decision_small.dtype)
    decision[
        :decision_small.shape[0],
        :decision_small.shape[1]
    ] = decision_small

    safe_remove  = (decision == 1)
    safe_protect = (decision == 2)

    require(not np.any(safe_remove & ~m3),
            f'{page}: SAFE_REMOVE is not a subset of M3.')
    require(not np.any(safe_protect & ~m3),
            f'{page}: SAFE_PROTECT is not a subset of M3.')
    require(not np.any(safe_remove & safe_protect),
            f'{page}: SAFE_REMOVE/SAFE_PROTECT overlap.')
    
    was_padded = (decision_small.shape != m3.shape)

    return safe_remove, safe_protect, p, decision_small.shape, was_padded

mask_rows = []

for page in pages:
    sr, sp, path, n50_shape, was_padded = resolve_m5_decisions(page)

    m3 = page_masks[page]['M3']

    # Both SAFE_REMOVE and SAFE_PROTECT leave the residual
    # reconstruction target. Everything else remains exact M3.
    m5 = m3 & ~sr & ~sp

    page_masks[page]['SAFE_REMOVE']  = sr
    page_masks[page]['SAFE_PROTECT'] = sp
    page_masks[page]['M5'] = m5

    mask_rows.append({
        'page': page,
        'source_group': str(
            regions.loc[regions.page == page, 'source_group'].iloc[0]
        ),
        'M3_pixels': int(m3.sum()),
        'SAFE_REMOVE_pixels': int(sr.sum()),
        'SAFE_PROTECT_pixels': int(sp.sum()),
        'M5_pixels': int(m5.sum()),
        'M4_pixels': int(page_masks[page]['M4_DIAGNOSTIC'].sum()),
        'safe_remove_subset_m3': bool(not np.any(sr & ~m3)),
        'safe_protect_subset_m3': bool(not np.any(sp & ~m3)),
        'm5_decision_path': str(path),
        'm5_decision_sha256': sha(path),
        'notebook50_decision_shape': str(tuple(n50_shape)),
        'authoritative_page_shape': str(tuple(m3.shape)),
        'notebook50_decision_padded': bool(was_padded),
    })

mask_df = pd.DataFrame(mask_rows)
atomic_csv(mask_df, TABLES/'mask_arm_inventory.csv')

require(mask_df.safe_remove_subset_m3.all(),
        'SAFE_REMOVE subset integrity failed.')
require(mask_df.safe_protect_subset_m3.all(),
        'SAFE_PROTECT subset integrity failed.')

print(
    mask_df[
        ['M3_pixels',
         'SAFE_REMOVE_pixels',
         'SAFE_PROTECT_pixels',
         'M5_pixels',
         'M4_pixels']
    ].sum()
)

## 6. Exact reconstruction functions


In [ ]:
def reconstruct_classical(obs,mask,method):
    m=(mask>0).astype(np.uint8)*255
    if not m.any(): return obs.copy()
    if method=='R2_telea': cand=cv2.inpaint(obs,m,3,cv2.INPAINT_TELEA)
    elif method=='R1_biharmonic': cand=np.clip(sk_inpaint.inpaint_biharmonic(obs.astype(np.float32)/255.,m>0,channel_axis=None)*255,0,255).astype(np.uint8)
    else: raise ValueError(method)
    out=obs.copy(); out[m>0]=cand[m>0]; require(np.array_equal(out[m==0],obs[m==0]),'Classical restorer outside-mask drift.'); return out

def centred_patch_geometry(shape,y0,y1,x0,x1,size=PATCH):
    h,w=map(int,shape[:2]); cy=(int(y0)+int(y1))//2; cx=(int(x0)+int(x1))//2; half=size//2
    ny0=cy-half; nx0=cx-half; ny1=ny0+size; nx1=nx0+size
    py0=max(0,ny0); px0=max(0,nx0); py1=min(h,ny1); px1=min(w,nx1)
    qy0=py0-ny0; qx0=px0-nx0; qy1=qy0+(py1-py0); qx1=qx0+(px1-px0)
    return {'page':(slice(py0,py1),slice(px0,px1)),'patch':(slice(qy0,qy1),slice(qx0,qx1))}

def crop_pad(a,y0,y1,x0,x1,size=PATCH,pad_mode='reflect'):
    g=centred_patch_geometry(a.shape,y0,y1,x0,x1,size); core=a[g['page']]; py0=g['patch'][0].start; px0=g['patch'][1].start; py1=size-g['patch'][0].stop; px1=size-g['patch'][1].stop
    mode=pad_mode
    if mode=='reflect' and (core.shape[0]<2 or core.shape[1]<2): mode='edge'
    if mode=='constant': out=np.pad(core,((py0,py1),(px0,px1)),mode='constant',constant_values=0)
    else: out=np.pad(core,((py0,py1),(px0,px1)),mode=mode)
    require(out.shape==(size,size),f'crop_pad shape {out.shape}'); return out

def run_learned(model,obs,mask,r):
    y0,y1,x0,x1=map(int,[r.y0,r.y1,r.x0,r.x1]); crop=crop_pad(obs,y0,y1,x0,x1); mc=crop_pad(mask.astype(np.uint8),y0,y1,x0,x1,pad_mode='constant')>0
    paper=int(np.median(crop[crop>200])) if np.any(crop>200) else 255; corrupted=crop.copy(); corrupted[mc]=paper
    x=np.stack([corrupted.astype(np.float32)/255.,mc.astype(np.float32)]).astype(np.float32)
    with torch.no_grad(): pred=(model(torch.from_numpy(x)[None].to(DEVICE))[0,0].cpu().numpy()*255).clip(0,255).astype(np.uint8)
    return pred

def place_prediction(original,pred_patch,full_mask,r):
    g=centred_patch_geometry(original.shape,int(r.y0),int(r.y1),int(r.x0),int(r.x1),PATCH); out=original.copy(); lm=full_mask[g['page']]; lp=pred_patch[g['patch']]; lo=out[g['page']]
    require(lo.shape==lm.shape==lp.shape,f'Placement mismatch {r.region_id}'); lo[lm]=lp[lm]; out[g['page']]=lo
    require(np.array_equal(out[~full_mask],original[~full_mask]),f'{r.region_id}: learned outside-mask drift'); return out


## A. FINAL Stage-F Forensic Experiment - STEP 1

In [ ]:
# ============================================================
# FINAL STAGE-F FORENSIC EXPERIMENT
# Cell 1 of 2: install cv2.inpaint capture hook
#
# Run this immediately BEFORE Notebook51R2's original
# candidate-construction cell.
#
# Then run the ORIGINAL Notebook51R2 candidate cell unchanged.
# Do not paste reconstructed Notebook53 logic in its place.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import inspect
import json
import platform
import sys

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

ROOT = Path(
    '/content/drive/MyDrive/MusicAnnotationProject'
)

CAPTURE_ROOT = (
    ROOT /
    'stage_f_final_forensic_capture' /
    (
        datetime.now().strftime('%Y%m%d_%H%M%S')
        +
        '_notebook51r2_actual_calls'
    )
)

CALLS = CAPTURE_ROOT / 'calls'
TABLES_CAPTURE = CAPTURE_ROOT / 'tables'

for directory in [
    CAPTURE_ROOT,
    CALLS,
    TABLES_CAPTURE,
]:
    directory.mkdir(
        parents=True,
        exist_ok=False,
    )


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def stage_f_array_hash(array):
    array = np.ascontiguousarray(array)

    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode())
    digest.update(str(array.shape).encode())
    digest.update(array.tobytes())

    return digest.hexdigest()


def stage_f_write_array(path, array):
    path = Path(path)

    np.save(
        path,
        np.ascontiguousarray(array),
        allow_pickle=False,
    )


def stage_f_write_preview(path, array):
    path = Path(path)

    value = np.asarray(array)

    if value.ndim != 2:
        return

    if value.dtype == bool:
        preview = (
            value.astype(np.uint8)
            * 255
        )

    elif value.dtype == np.uint8:
        preview = value

    else:
        finite = np.isfinite(value)

        if not finite.any():
            return

        low = float(
            np.nanmin(value)
        )

        high = float(
            np.nanmax(value)
        )

        if high <= low:
            preview = np.zeros(
                value.shape,
                dtype=np.uint8,
            )
        else:
            preview = np.clip(
                (
                    value.astype(np.float64)
                    -
                    low
                )
                /
                (high - low)
                *
                255,
                0,
                255,
            ).astype(np.uint8)

    cv2.imwrite(
        str(path),
        preview,
    )


def stage_f_safe_scalar(value):
    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return str(value)

    if isinstance(value, Path):
        return str(value)

    return None


def stage_f_stack_context():
    """
    Inspect the live Notebook51R2 call stack for useful page,
    region, arm and restorer variables.

    This is diagnostic metadata only. Array identity does not
    depend on whether these names are successfully recovered.
    """

    context = {}

    interesting_names = {
        'page',
        'page_id',
        'page_name',
        'region',
        'region_id',
        'rid',
        'arm',
        'rest',
        'restorer',
        'method',
        'candidate_name',
    }

    frames = inspect.stack()

    try:
        for frame_info in frames[2:20]:
            local_values = frame_info.frame.f_locals

            for name in interesting_names:
                if (
                    name in local_values
                    and name not in context
                ):
                    scalar = stage_f_safe_scalar(
                        local_values[name]
                    )

                    if scalar is not None:
                        context[name] = scalar

            if 'regional' in local_values:
                regional = local_values['regional']

                if isinstance(
                    regional,
                    np.ndarray,
                ):
                    context.setdefault(
                        'regional_pixels',
                        str(
                            int(
                                np.asarray(
                                    regional
                                ).astype(bool).sum()
                            )
                        ),
                    )

                    context.setdefault(
                        'regional_hash',
                        stage_f_array_hash(
                            np.asarray(regional)
                        ),
                    )

            if 'm5' in local_values:
                m5_value = local_values['m5']

                if isinstance(
                    m5_value,
                    np.ndarray,
                ):
                    context.setdefault(
                        'm5_pixels',
                        str(
                            int(
                                np.asarray(
                                    m5_value
                                ).astype(bool).sum()
                            )
                        ),
                    )

                    context.setdefault(
                        'm5_hash',
                        stage_f_array_hash(
                            np.asarray(m5_value)
                        ),
                    )

    finally:
        del frames

    return context


# ------------------------------------------------------------
# Preserve the real OpenCV function
# ------------------------------------------------------------

if hasattr(
    cv2,
    '_stage_f_original_inpaint',
):
    ORIGINAL_INPAINT = (
        cv2._stage_f_original_inpaint
    )
else:
    ORIGINAL_INPAINT = cv2.inpaint

    cv2._stage_f_original_inpaint = (
        ORIGINAL_INPAINT
    )


STAGE_F_CAPTURE_ROWS = []
STAGE_F_CALL_NUMBER = 0


# ------------------------------------------------------------
# Capture wrapper
# ------------------------------------------------------------

def stage_f_captured_inpaint(
    source,
    mask,
    radius,
    flags,
):
    global STAGE_F_CALL_NUMBER

    STAGE_F_CALL_NUMBER += 1
    call_number = STAGE_F_CALL_NUMBER

    source_array = np.asarray(source)
    mask_array = np.asarray(mask)

    before_source = source_array.copy()
    before_mask = mask_array.copy()

    result = ORIGINAL_INPAINT(
        source,
        mask,
        radius,
        flags,
    )

    result_array = np.asarray(result)

    context = stage_f_stack_context()

    page = (
        context.get('page')
        or context.get('page_id')
        or context.get('page_name')
        or 'UNKNOWN_PAGE'
    )

    region = (
        context.get('region_id')
        or context.get('region')
        or context.get('rid')
        or 'UNKNOWN_REGION'
    )

    restorer = (
        context.get('restorer')
        or context.get('rest')
        or context.get('method')
        or 'UNKNOWN_RESTORER'
    )

    call_name = (
        f'call_{call_number:06d}'
        f'__{str(page)}'
        f'__{str(region)}'
    )

    call_name = (
        call_name
        .replace('/', '_')
        .replace('\\', '_')
        .replace(':', '_')
        .replace(' ', '_')
    )

    call_directory = (
        CALLS /
        call_name
    )

    call_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    stage_f_write_array(
        call_directory /
        'source.npy',
        before_source,
    )

    stage_f_write_array(
        call_directory /
        'mask.npy',
        before_mask,
    )

    stage_f_write_array(
        call_directory /
        'result.npy',
        result_array,
    )

    stage_f_write_preview(
        call_directory /
        'source.png',
        before_source,
    )

    stage_f_write_preview(
        call_directory /
        'mask.png',
        before_mask > 0,
    )

    stage_f_write_preview(
        call_directory /
        'result.png',
        result_array,
    )

    masked = (
        before_mask > 0
    )

    changed_outside_mask = int(
        np.sum(
            result_array[~masked]
            !=
            before_source[~masked]
        )
    )

    metadata = {
        'call_number':
            call_number,

        'page':
            str(page),

        'region':
            str(region),

        'restorer':
            str(restorer),

        'radius':
            float(radius),

        'flags':
            int(flags),

        'flags_name':
            (
                'INPAINT_TELEA'
                if int(flags)
                == int(cv2.INPAINT_TELEA)
                else
                (
                    'INPAINT_NS'
                    if int(flags)
                    == int(cv2.INPAINT_NS)
                    else
                    str(flags)
                )
            ),

        'source_shape':
            list(
                before_source.shape
            ),

        'source_dtype':
            str(
                before_source.dtype
            ),

        'mask_shape':
            list(
                before_mask.shape
            ),

        'mask_dtype':
            str(
                before_mask.dtype
            ),

        'result_shape':
            list(
                result_array.shape
            ),

        'result_dtype':
            str(
                result_array.dtype
            ),

        'mask_pixels':
            int(
                masked.sum()
            ),

        'source_array_sha256':
            stage_f_array_hash(
                before_source
            ),

        'mask_array_sha256':
            stage_f_array_hash(
                before_mask
            ),

        'mask_boolean_sha256':
            stage_f_array_hash(
                masked.astype(
                    np.uint8
                )
            ),

        'result_array_sha256':
            stage_f_array_hash(
                result_array
            ),

        'result_changed_outside_mask':
            changed_outside_mask,

        'context':
            context,

        'call_directory':
            str(call_directory),
    }

    (
        call_directory /
        'call_metadata.json'
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
            default=str,
        )
    )

    STAGE_F_CAPTURE_ROWS.append(
        metadata
    )

    print(
        'CAPTURED',
        call_number,
        'page',
        page,
        'region',
        region,
        'restorer',
        restorer,
        'shape',
        before_source.shape,
        'mask',
        int(masked.sum()),
    )

    return result


# ------------------------------------------------------------
# Install hook
# ------------------------------------------------------------

cv2.inpaint = stage_f_captured_inpaint


# ------------------------------------------------------------
# Environment record
# ------------------------------------------------------------

environment = {
    'created_utc':
        datetime.now(
            timezone.utc
        ).isoformat(),

    'python':
        sys.version,

    'opencv':
        cv2.__version__,

    'numpy':
        np.__version__,

    'pandas':
        pd.__version__,

    'platform':
        platform.platform(),

    'capture_root':
        str(CAPTURE_ROOT),

    'instruction':
        (
            'Run the original Notebook51R2 '
            'candidate-construction cell unchanged, '
            'then run the finalisation cell.'
        ),
}

(
    CAPTURE_ROOT /
    'software_environment.json'
).write_text(
    json.dumps(
        environment,
        indent=2,
    )
)

print('\nCapture hook installed.')
print('Capture root:', CAPTURE_ROOT)
print(
    '\nNow run the ORIGINAL Notebook51R2 '
    'candidate-construction cell unchanged.'
)

## 7. Candidate construction, page-checkpointed

M3 and M4 are copied from Notebook 46R6's locked whole-page candidates. M5-REMOVE is the only newly constructed arm. For M5, each Notebook 45 region receives the page-global M5 mask intersected with that region's M3 footprint, preserving the same region geometry and held-out source-group model assignment used by Notebook 46R6.


In [ ]:
# Build lookup for frozen Notebook46 M3/M4 candidates.
c46=pd.read_csv(N46/'tables/whole_page_candidate_inventory.csv')
c46['page']=c46.page.astype(str)

def frozen46(page,arm,rest):
    q=c46[(c46.page==page)&(c46.arm==arm)&(c46.restorer==rest)]
    require(len(q)==1,f'{page}/{arm}/{rest}: expected one Notebook46 whole-page candidate, got {len(q)}')
    p=Path(str(q.iloc[0].candidate)); require(p.exists(),f'Missing Notebook46 candidate {p}'); return p

def checkpoint_path(page): return CHECK/f'{page}.json'
def valid_checkpoint(page,contract):
    p=checkpoint_path(page)
    if not p.exists(): return False
    try:
        x=json.loads(p.read_text());
        if x.get('contract')!=contract or x.get('complete') is not True: return False
        return all(Path(r['path']).exists() and sha(Path(r['path']))==r['sha256'] for r in x['outputs'])
    except Exception: return False

contract=hashlib.sha256(json.dumps({'prereg':PREREG,'n45_lock':sha(N45/'mask_lock.json'),'n46_verdict':sha(N46/'v8_6_verdict.json'),'n46_pre':sha(N46/'restorer_pre_registration.json'),'n46_models':sha(N46/'audit/learned_model_inventory.csv'),'n50_verdict':sha(N50/'notebook50_verdict.json'),'n50_inventory':sha(N50/'tables/m5_map_inventory.csv')},sort_keys=True).encode()).hexdigest()
status=[]
for page in pages:
    if valid_checkpoint(page,contract): status.append({'page':page,'status':'reused'}); continue
    obs=page_obs[page]; auth=page_auth[page]; protected=page_protected[page]; page_dir=CAND/page; page_dir.mkdir(parents=True,exist_ok=True); outrows=[]
    # Reuse frozen M3 / M4 whole-page candidates exactly.
    for arm46,arm51 in [('M3','M3'),('M4','M4_DIAGNOSTIC')]:
        for rest in RESTORERS:
            src=frozen46(page,arm46,rest); dst=page_dir/f'{arm51}_{rest}.png'; shutil.copy2(src,dst); im=cv2.imread(str(dst),0)
            require(im is not None and im.shape==obs.shape,f'{page}/{arm51}/{rest}: unreadable frozen candidate'); require(np.array_equal(im[~auth],obs[~auth]),f'{page}/{arm51}/{rest}: frozen outside-mask drift')
            outrows.append({'arm':arm51,'restorer':rest,'path':str(dst),'sha256':sha(dst),'source':'Notebook46R6_frozen'})
    # Construct M5-REMOVE from the same base/protected whole-page contract.
    m5=page_masks[page]['M5']; candidate_base=page_base[page].copy(); candidate_base[protected]=obs[protected]
    group=str(regions.loc[regions.page==page,'source_group'].iloc[0]); model=load_n46_model(checkpoint_by_group[group])
    pg=regions[regions.page==page]
    for rest in RESTORERS:
        candidate=candidate_base.copy()
        for _,r in pg.iterrows():
            # Restrict page-global M5 to the region's original M3 footprint. This preserves Notebook46 regional ownership/placement semantics.
            mp=N45/'masks'/page/str(r.region_id).replace(':','_')/'M3.png'; rm=cv2.imread(str(mp),0); require(rm is not None and rm.shape==obs.shape,f'Invalid regional M3 mask {mp}')
            regional=m5 & (rm>0)
            if not regional.any(): continue
            if rest=='R3_learned_unet': pred=run_learned(model,obs,regional,r); recon=place_prediction(obs,pred,regional,r)
            else: recon=reconstruct_classical(obs,regional,rest)
            candidate[regional]=recon[regional]
        require(np.array_equal(candidate[~auth],obs[~auth]),f'{page}/M5/{rest}: outside-authority drift')
        require(np.array_equal(candidate[protected & ~m5],obs[protected & ~m5]),f'{page}/M5/{rest}: protected non-residual drift')
        dst=page_dir/f'M5_{rest}.png'; tmp=dst.with_suffix('.tmp.png'); require(cv2.imwrite(str(tmp),candidate),f'Cannot write {tmp}'); tmp.replace(dst)
        outrows.append({'arm':'M5','restorer':rest,'path':str(dst),'sha256':sha(dst),'source':'Notebook51R2_new'})
    atomic_json(checkpoint_path(page),{'version':'51R2-page-v1','page':page,'complete':True,'contract':contract,'outputs':outrows,'created_utc':datetime.now(timezone.utc).isoformat()}); status.append({'page':page,'status':'computed'})
    del model; gc.collect(); torch.cuda.empty_cache()
atomic_csv(pd.DataFrame(status),AUDIT/'page_completion.csv'); print(pd.DataFrame(status).status.value_counts())


## B. FINAL Stage-F Forensic Experiment - STEP 2

In [ ]:
# ============================================================
# FINAL STAGE-F FORENSIC EXPERIMENT
# Cell 2 of 2: finalise captured calls and compare the
# actual Notebook51R2 rerun with its durable golden masters
# ============================================================

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

required_names = [
    'CAPTURE_ROOT',
    'TABLES_CAPTURE',
    'STAGE_F_CAPTURE_ROWS',
    'ORIGINAL_INPAINT',
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        'Run the capture-hook cell first. Missing: '
        f'{missing_names}'
    )

if len(STAGE_F_CAPTURE_ROWS) == 0:
    raise RuntimeError(
        'No cv2.inpaint calls were captured. '
        'The original Notebook51R2 candidate cell was '
        'not run after installing the hook, or it did '
        'not call cv2.inpaint.'
    )


# ------------------------------------------------------------
# Restore OpenCV immediately
# ------------------------------------------------------------

cv2.inpaint = ORIGINAL_INPAINT

print(
    'Restored original cv2.inpaint.'
)


# ------------------------------------------------------------
# Write captured call inventory
# ------------------------------------------------------------

call_rows = []

for row in STAGE_F_CAPTURE_ROWS:
    flattened = {
        key: value
        for key, value in row.items()
        if key != 'context'
    }

    context = row.get(
        'context',
        {}
    )

    for key, value in context.items():
        flattened[
            f'context_{key}'
        ] = value

    call_rows.append(
        flattened
    )

call_df = pd.DataFrame(
    call_rows
)

call_inventory_path = (
    TABLES_CAPTURE /
    'captured_inpaint_calls.csv'
)

call_df.to_csv(
    call_inventory_path,
    index=False,
)


# ------------------------------------------------------------
# Resolve historical Notebook51R2 inventory
# ------------------------------------------------------------

if 'N51' not in globals():
    raise RuntimeError(
        'N51 is not defined. Set N51 to the durable '
        'historical Notebook51R2 run before running '
        'this finalisation cell.'
    )

N51 = Path(N51)

historical_inventory_hits = sorted(
    path
    for path in N51.rglob(
        'candidate_inventory.csv'
    )
    if path.is_file()
)

if len(historical_inventory_hits) != 1:
    raise RuntimeError(
        'Expected one historical Notebook51R2 '
        'candidate_inventory.csv, found '
        f'{len(historical_inventory_hits)}:\n'
        f'{historical_inventory_hits}'
    )

historical_inventory_path = (
    historical_inventory_hits[0]
)

historical_inventory = pd.read_csv(
    historical_inventory_path,
    dtype=str,
    keep_default_na=False,
)

required_columns = {
    'page',
    'arm',
    'restorer',
    'candidate',
}

missing_columns = (
    required_columns
    -
    set(
        historical_inventory.columns
    )
)

if missing_columns:
    raise RuntimeError(
        'Historical candidate inventory lacks: '
        f'{sorted(missing_columns)}'
    )

for column in required_columns:
    historical_inventory[column] = (
        historical_inventory[column]
        .astype(str)
        .str.strip()
    )

historical_golden = historical_inventory[
    (
        historical_inventory.arm
        ==
        'M5_REMOVE'
    )
    &
    (
        historical_inventory.restorer
        ==
        'R2_telea'
    )
].copy()

if historical_golden.empty:
    raise RuntimeError(
        'Historical inventory contains no '
        'M5_REMOVE/R2_telea rows.'
    )

if historical_golden.page.duplicated().any():
    raise RuntimeError(
        'Historical golden-master inventory contains '
        'duplicate pages.'
    )


# ------------------------------------------------------------
# Candidate roots produced by the actual rerun
# ------------------------------------------------------------

candidate_roots = []

for variable_name in [
    'CAND',
    'CANDIDATES',
    'CANDIDATE_DIR',
    'OUT',
    'RUN',
    'OUTPUT',
]:

    if variable_name not in globals():
        continue

    try:
        value = Path(
            globals()[variable_name]
        )
    except Exception:
        continue

    if value.exists():
        candidate_roots.append(
            (
                variable_name,
                value,
            )
        )

# Deduplicate.
unique_roots = []

seen_roots = set()

for variable_name, path in candidate_roots:
    resolved = str(
        path.resolve()
    )

    if resolved in seen_roots:
        continue

    seen_roots.add(
        resolved
    )

    unique_roots.append(
        (
            variable_name,
            path,
        )
    )

candidate_roots = unique_roots

print('\nPossible actual-rerun roots:')

for variable_name, path in candidate_roots:
    print(
        f'  {variable_name}: {path}'
    )


# ------------------------------------------------------------
# Resolve one newly generated M5_REMOVE/R2_telea candidate
# for each page.
# ------------------------------------------------------------

def semantic_hash(array):
    array = np.ascontiguousarray(
        array
    )

    digest = hashlib.sha256()
    digest.update(
        str(array.dtype).encode()
    )
    digest.update(
        str(array.shape).encode()
    )
    digest.update(
        array.tobytes()
    )

    return digest.hexdigest()


def resolve_historical_reference(row):
    recorded = Path(
        row['candidate']
    )

    if recorded.exists():
        return recorded

    fallback = (
        N51 /
        '03_candidates' /
        str(row['page']) /
        recorded.name
    )

    if fallback.exists():
        return fallback

    hits = sorted(
        path
        for path in N51.rglob(
            recorded.name
        )
        if (
            path.is_file()
            and path.parent.name
            ==
            str(row['page'])
        )
    )

    if len(hits) != 1:
        raise RuntimeError(
            f'{row["page"]}: cannot resolve historical '
            f'golden master. Hits: {hits}'
        )

    return hits[0]


def resolve_rerun_candidate(
    page,
    filename,
):
    hits = []

    for _, root in candidate_roots:
        hits.extend(
            path
            for path in root.rglob(
                filename
            )
            if (
                path.is_file()
                and path..parent.name
                ==
                str(page)
            )
        )

    # Exclude files underneath the historical N51 tree.
    n51_resolved = N51.resolve()

    filtered = []

    for path in hits:
        try:
            path.resolve().relative_to(
                n51_resolved
            )

            # Historical reference, not the new rerun.
            continue

        except ValueError:
            filtered.append(path)

    hits = sorted(
        set(filtered)
    )

    if len(hits) != 1:
        raise RuntimeError(
            f'{page}: expected one newly generated '
            f'{filename}; found {len(hits)}:\n{hits}\n'
           'If the original Notebook51R2 cell writes '
            'beneath N51 itself, use a fresh output root '
            'and rerun the experiment.'
        )

    return hits[0]


comparison_rows = []
difference_coordinates = []

for _, historical_row in (
    historical_golden.iterrows()
):

    page = str(
        historical_row.page
    )

    historical_path = (
        resolve_historical_reference(
            historical_row
        )
    )

    rerun_path = resolve_rerun_candidate(
        page,
        Path(
            historical_row.candidate
        ).name,
    )

    historical = cv2.imread(
        str(historical_path),
        cv2.IMREAD_GRAYSCALE,
    )

    rerun = cv2.imread(
        str(rerun_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if (
        historical is None
        or rerun is None
    ):
        raise RuntimeError(
            f'{page}: unreadable historical/rerun image.'
        )

    if historical.shape != rerun.shape:
        raise RuntimeError(
            f'{page}: shape mismatch: historical '
            f'{historical.shape}, rerun {rerun.shape}.'
        )

    different = (
        historical != rerun
    )

    difference_count = int(
        different.sum()
    )

    absolute_difference = np.abs(
        historical.astype(np.int16)
        -
        rerun.astype(np.int16)
    )

    comparison_rows.append({
        'page':
            page,

        'historical_path':
            str(historical_path),

        'rerun_path':
            str(rerun_path),

        'exact':
            bool(
                difference_count == 0
            ),

        'different_pixels':
            difference_count,

        'maximum_absolute_difference':
            int(
                absolute_difference.max()
            ),

        'historical_array_sha256':
            semantic_hash(
                historical
            ),

        'rerun_array_sha256':
            semantic_hash(
                rerun
            ),
    })

    ys, xs = np.where(
        different
    )

    for y, x in zip(ys, xs):
        difference_coordinates.append({
            'page':
                page,

            'y':
                int(y),

            'x':
                int(x),

            'historical_value':
                int(
                    historical[y, x]
                ),

            'rerun_value':
                int(
                    rerun[y, x]
                ),

            'absolute_difference':
                int(
                    absolute_difference[y, x]
                ),
        })


comparison_df = pd.DataFrame(
    comparison_rows
)

coordinate_df = pd.DataFrame(
    difference_coordinates
)

comparison_path = (
    TABLES_CAPTURE /
    'notebook51r2_actual_rerun_vs_historical.csv'
)

coordinate_path = (
    TABLES_CAPTURE /
    'notebook51r2_actual_rerun_difference_coordinates.csv'
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

coordinate_df.to_csv(
    coordinate_path,
    index=False,
)


# ------------------------------------------------------------
# Final classification
# ------------------------------------------------------------

all_exact = bool(
    comparison_df.exact.all()
)

total_difference = int(
    comparison_df.different_pixels.sum()
)

failed_pages = (
    comparison_df.loc[
        ~comparison_df.exact,
        'page'
    ]
    .astype(str)
    .tolist()
)

captured_telea_calls = int(
    (
        call_df.flags_name
        ==
        'INPAINT_TELEA'
    ).sum()
)

if all_exact:
    diagnosis = (
        'ACTUAL_NOTEBOOK51R2_SOURCE_REPRODUCES_'
        'HISTORICAL_GOLDEN_MASTER'
    )

    required_action = (
        'Compare the captured source.npy and mask.npy '
        'arrays from the failing 53R2 pages with the '
        'corresponding 53R2 Telea call arrays. The first '
        'hash difference is the precise implementation '
        'change required for Stage F.'
    )

else:
    diagnosis = (
        'ACTUAL_NOTEBOOK51R2_SOURCE_DOES_NOT_'
        'REPRODUCE_HISTORICAL_GOLDEN_MASTER'
    )

    required_action = (
        'Do not continue changing Notebook53R2 logic. '
        'The historical golden master depends on a '
        'different software/runtime or unrecorded state. '
        'Recover the historical environment or create a '
        'separately governed reference-locked replay path.'
    )


verdict = {
    'experiment':
        'Actual Notebook51R2 reconstruction-call capture '
        'and historical golden-master replay',

    'capture_root':
        str(CAPTURE_ROOT),

    'captured_inpaint_calls':
        int(
            len(call_df)
        ),

    'captured_telea_calls':
        captured_telea_calls,

    'historical_pages':
        int(
            len(comparison_df)
        ),

    'historical_rerun_exact_pages':
        int(
            comparison_df.exact.sum()
        ),

    'historical_rerun_failed_pages':
        failed_pages,

    'historical_rerun_total_different_pixels':
        total_difference,

    'actual_notebook51r2_source_reproduces_golden':
        all_exact,

    'diagnosis':
        diagnosis,

    'required_action':
        required_action,

    'stage_F_may_be_declared_pass':
        False,

    'note':
        (
            'This experiment identifies the route to '
            'Stage F equivalence. Stage F passes only '
            'when Notebook53R2 itself reaches zero '
            'different corrected-page pixels.'
        ),
}

verdict_path = (
    CAPTURE_ROOT /
    'final_stage_f_forensic_verdict.json'
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
    )
)

print('\n' + '=' * 80)
print('FINAL STAGE-F FORENSIC VERDICT')
print('=' * 80)

print(
    json.dumps(
        verdict,
        indent=2,
    )
)

print('\nPage comparison:')

display(
    comparison_df.sort_values(
        [
            'different_pixels',
            'page',
        ],
        ascending=[
            False,
            True,
        ],
    )
)

print('\nOutputs:')
print(' ', call_inventory_path)
print(' ', comparison_path)
print(' ', coordinate_path)
print(' ', verdict_path)
print(' ', CALLS)

## 8. Candidate lock before clean truth


In [ ]:
rows=[]
for page in pages:
    sg=str(regions.loc[regions.page==page,'source_group'].iloc[0])
    for arm in ARMS:
        for rest in RESTORERS:
            p=CAND/page/f'{arm}_{rest}.png'; require(p.exists(),f'Missing candidate {p}'); rows.append({'page':page,'source_group':sg,'arm':arm,'restorer':rest,'candidate':str(p),'sha256':sha(p),'array_sha256':array_hash(cv2.imread(str(p),0))})
candidate_inventory=pd.DataFrame(rows).sort_values(['page','arm','restorer']).reset_index(drop=True); atomic_csv(candidate_inventory,LOCK/'candidate_inventory.csv')
lock_id=hashlib.sha256(candidate_inventory.to_csv(index=False).encode()).hexdigest(); atomic_json(LOCK/'candidate_lock.json',{'version':'51R2-candidate-lock-v1','lock_id':lock_id,'contract':contract,'rows':len(candidate_inventory),'truth_loaded':False,'created_utc':datetime.now(timezone.utc).isoformat()})
print('CANDIDATE LOCKED:',lock_id,'rows:',len(candidate_inventory))


## 9. Unlock clean truth and score fixed authoritative-mask region


In [ ]:
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
require(set(pages).issubset(clean_map),f'Missing clean pages after candidate lock: {sorted(set(pages)-set(clean_map))}')

def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0
def f1(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1); r=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum(); return float(p),float(r),float(2*p*r/max(p+r,1e-12))

score_rows=[]
for _,row in candidate_inventory.iterrows():
    page=row.page; clean=cv2.imread(str(clean_map[page]),0); pred=cv2.imread(str(row.candidate),0); obs=page_obs[page]; auth=page_auth[page]
    require(clean is not None and pred is not None and clean.shape==pred.shape==auth.shape,f'{page}: scoring shape/read failure')
    diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32)); _,sm=ssim(clean,pred,data_range=255,full=True); ip,ir,if1=f1(ink(clean),ink(pred),auth); ep,er,ef1=f1(edges(clean),edges(pred),auth)
    score_rows.append({'page':page,'source_group':row.source_group,'arm':row.arm,'restorer':row.restorer,'authoritative_pixels':int(auth.sum()),'mae':float(diff[auth].mean()),'ssim':float(sm[auth].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'false_inserted_ink':int((ink(pred)&~ink(clean)&auth).sum()),'missing_ink':int((ink(clean)&~ink(pred)&auth).sum()),'outside_mask_drift':int(np.sum(pred[~auth]!=obs[~auth]))})
metrics=pd.DataFrame(score_rows); require((metrics.outside_mask_drift==0).all(),'Outside-authoritative-mask drift detected in scored candidates.'); atomic_csv(metrics,TABLES/'candidate_metrics.csv')
summary=metrics.groupby(['arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ssim=('ssim','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'),outside_mask_drift=('outside_mask_drift','sum'))
atomic_csv(summary,TABLES/'candidate_summary.csv'); display(summary)


## 10. Paired page-clustered M5-REMOVE vs M3 inference


In [ ]:
def benefit(metric,m5,m3): return (m3-m5) if metric in ['mae','false_inserted_ink','missing_ink'] else (m5-m3)
def boot_ci(v,n=BOOT,seed=SEED):
    v=np.asarray(v,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); b=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(n)]); return float(v.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

paired=[]
for rest,g in metrics.groupby('restorer'):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric); d=benefit(metric,p.M5,p.M3); mean,lo,hi=boot_ci(d)
        paired.append({'restorer':rest,'metric':metric,'benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum()),'pages':int(d.notna().sum())})
paired_df=pd.DataFrame(paired); atomic_csv(paired_df,TABLES/'paired_m5_remove_vs_m3_bootstrap.csv'); display(paired_df)

source=[]
for (sg,rest),g in metrics.groupby(['source_group','restorer']):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric); d=benefit(metric,p.M5,p.M3)
        source.append({'source_group':sg,'restorer':rest,'metric':metric,'pages':int(len(d)),'benefit_mean':float(d.mean()),'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
source_df=pd.DataFrame(source); atomic_csv(source_df,TABLES/'m5_remove_vs_m3_by_source_group.csv')


## 11. Restorer-gap contraction


In [ ]:
gap_rows=[]
for (page,arm),g in metrics.groupby(['page','arm']):
    row={'page':page,'source_group':g.source_group.iloc[0],'arm':arm}
    for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        v=g[metric].to_numpy(float); row[f'{metric}_range']=float(np.nanmax(v)-np.nanmin(v)); row[f'{metric}_sd']=float(np.nanstd(v))
    gap_rows.append(row)
gaps=pd.DataFrame(gap_rows); atomic_csv(gaps,TABLES/'restorer_gap_by_page.csv')
contract_rows=[]
for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
    for stat in ['range','sd']:
        p=gaps.pivot(index='page',columns='arm',values=f'{metric}_{stat}'); d=p.M3-p.M5; mean,lo,hi=boot_ci(d)
        contract_rows.append({'metric':metric,'gap_stat':stat,'contraction_mean':mean,'ci95_low':lo,'ci95_high':hi,'pages_contracting':int((d>0).sum()),'pages_expanding':int((d<0).sum()),'ties':int((d==0).sum())})
contraction=pd.DataFrame(contract_rows); atomic_csv(contraction,TABLES/'restorer_gap_contraction.csv'); display(contraction)


## 12. Diagnostic M4 positioning


In [ ]:
# M4 is not a promotion target. Report distance from M3/M5 to the diagnostic reference only.
rows=[]
for rest,g in metrics.groupby('restorer'):
    for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
        p=g.pivot(index='page',columns='arm',values=metric)
        higher=metric in ['ssim','ink_f1','edge_f1']
        # Positive values mean the first mask arm is better than the second.
        m5_m4=(p.M5-p.M4_DIAGNOSTIC)*(1 if higher else -1)
        m3_m4=(p.M3-p.M4_DIAGNOSTIC)*(1 if higher else -1)
        rows.append({'restorer':rest,'metric':metric,'M5_minus_M4_benefit_mean':float(m5_m4.mean()),'M3_minus_M4_benefit_mean':float(m3_m4.mean())})
atomic_csv(pd.DataFrame(rows),TABLES/'diagnostic_m4_positioning.csv')


## 13. Visual audit queues


In [ ]:
queue=[]
for rest,g in metrics.groupby('restorer'):
    p=g.set_index(['page','arm'])
    for page in pages:
        a=p.loc[(page,'M3')]; b=p.loc[(page,'M5')]
        queue.append({'page':page,'source_group':a.source_group,'restorer':rest,'edge_f1_benefit':b.edge_f1-a.edge_f1,'ink_f1_benefit':b.ink_f1-a.ink_f1,'mae_benefit':a.mae-b.mae,'false_inserted_benefit':a.false_inserted_ink-b.false_inserted_ink,'missing_ink_benefit':a.missing_ink-b.missing_ink})
queue=pd.DataFrame(queue); atomic_csv(queue,TABLES/'visual_audit_queue.csv')
for rest in RESTORERS:
    for label,q in [('best',queue[queue.restorer==rest].sort_values(['edge_f1_benefit','ink_f1_benefit'],ascending=False).head(10)),('worst',queue[queue.restorer==rest].sort_values(['edge_f1_benefit','ink_f1_benefit']).head(10))]:
        if q.empty: continue
        fig,axes=plt.subplots(len(q),3,figsize=(15,5*len(q)),squeeze=False)
        for i,row in enumerate(q.itertuples()):
            for j,(arm,title) in enumerate([('M3','M3'),('M5','M5'),('M4_DIAGNOSTIC','M4 diagnostic')]):
                im=cv2.imread(str(CAND/row.page/f'{arm}_{rest}.png'),0); axes[i,j].imshow(im,cmap='gray',vmin=0,vmax=255); axes[i,j].set_title(f'{row.page} {title}'); axes[i,j].axis('off')
        plt.tight_layout(); plt.savefig(VIS/f'{rest}_{label}_whole_pages.png',dpi=120,bbox_inches='tight'); plt.close(fig)
print('Visual audit:',VIS)


## 14. Fail-closed interpretation


In [ ]:
# Predeclared conservative research interpretation. Positive CI means M5 benefit.
struct=paired_df[paired_df.metric.isin(['ink_f1','edge_f1'])].copy()
positive_restorers=sorted(set(struct.loc[struct.ci95_low>0,'restorer']))
harmful=struct[struct.ci95_high<0].copy()
robust_contraction=contraction[contraction.ci95_low>0].copy()
if len(harmful): verdict='M5_REMOVE_RECONSTRUCTION_REGRESSION'
elif len(positive_restorers)>=2 and len(robust_contraction): verdict='M5_REMOVE_CAUSAL_GAIN_WITH_RESTORER_GAP_CONTRACTION'
elif len(positive_restorers)>=2: verdict='M5_REMOVE_CAUSAL_GAIN_WITHOUT_ROBUST_GAP_CONTRACTION'
else: verdict='M5_REMOVE_CAUSAL_GAIN_NOT_DEMONSTRATED'
record={'version':'8.7-notebook51R2','valid_run':True,'verdict':verdict,'development_only':True,'promotion_authority':False,'positive_structural_restorers':positive_restorers,'structural_harm_rows':harmful.to_dict('records'),'robust_gap_contraction_endpoints':robust_contraction[['metric','gap_stat','contraction_mean','ci95_low','ci95_high']].to_dict('records'),'m5_policy':'REMOVE-only','safe_protect_intervention':False,'m4_role':'diagnostic only'}
atomic_json(OUT/'notebook51_verdict.json',record); print(json.dumps(record,indent=2))


## 15. Identity and Final Upload Package


In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
atomic_json(OUT/'software_environment.json',environment)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook51R2','candidate_lock_id':lock_id,'contract':contract,'source_record':source_record,'mask_arms':ARMS,'restorers':RESTORERS,'heldout_unet_checkpoints':{g:{'path':str(p),'sha256':sha(p)} for g,p in checkpoint_by_group.items()},'pages':len(pages),'source_groups':int(regions.source_group.nunique()),'development_only':True,'promotion_authority':False,'software':environment}
atomic_json(OUT/'experiment_identity.json',identity)
lines=['V8.7 NOTEBOOK 51R2 - FROZEN M3 / M5-REMOVE / DIAGNOSTIC-M4 CROSSOVER','',f'Run: {OUT}',f'Pages: {len(pages)}',f'Source groups: {regions.source_group.nunique()}',f'Verdict: {verdict}','',f'Positive structural restorers: {positive_restorers}',f'Robust restorer-gap contraction endpoints: {len(robust_contraction)}','','M5 contains SAFE_REMOVE only. SAFE_PROTECT is not applied. M4 is diagnostic only. Clean truth was loaded only after the candidate lock. No production authority.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook51_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',OUT/'01_pre_registration.json',LOCK/'candidate_lock.json',TABLES/'mask_arm_inventory.csv',TABLES/'candidate_summary.csv',TABLES/'candidate_metrics.csv',TABLES/'paired_m5_remove_vs_m3_bootstrap.csv',TABLES/'m5_remove_vs_m3_by_source_group.csv',TABLES/'restorer_gap_contraction.csv',TABLES/'diagnostic_m4_positioning.csv',TABLES/'visual_audit_queue.csv',AUDIT/'page_completion.csv']
for pth in required: require(pth.exists(),f'Missing upload artefact {pth}')
for pth in required: shutil.copy2(pth,UPLOAD/pth.name)
(UPLOAD/'README.txt').write_text('Notebook 51R2 curated review package. Heavy candidate images and checkpoints remain in the durable run directory and are referenced by the identity/lock records.'+chr(10))
manifest=[]
for pth in sorted(UPLOAD.iterdir()):
    if pth.is_file(): manifest.append({'file':pth.name,'bytes':pth.stat().st_size,'sha256':sha(pth)})
atomic_csv(pd.DataFrame(manifest),UPLOAD/'upload_manifest.csv')
zip_path=UPLOAD/'notebook51R2_upload_bundle.zip'; tmp=UPLOAD/'notebook51R2_upload_bundle.tmp.zip'
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
    for pth in sorted(UPLOAD.iterdir()):
        if pth.is_file() and pth not in {zip_path,tmp}: z.write(pth,pth.name)
tmp.replace(zip_path)
print(chr(10).join(lines)); print('Bundle:',zip_path)


## Outputs to inspect first
1. `executive_summary.txt`
2. `notebook51_verdict.json`
3. `05_tables/paired_m5_remove_vs_m3_bootstrap.csv`
4. `05_tables/m5_remove_vs_m3_by_source_group.csv`
5. `05_tables/restorer_gap_contraction.csv`
6. `05_tables/candidate_summary.csv`
7. `06_visual_audit/`
8. `upload/notebook51R2_upload_bundle.zip`

If source paths have moved, set the explicit run override at the top. Do not weaken an integrity check merely to make the notebook continue.
